# Лабораторная работа 3 - Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.


## Шаг 1: Выбор датасета

Загружаем встроенный в seaborn датасет «Титаник». Выводим размер таблицы и количество пропусков, чтобы понять объём необходимой очистки. Выбираем признаки, которые будем использовать для предсказания выживания (survived).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, ShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = sns.load_dataset('titanic')
print("Размер датасета:", df.shape)
print("Пропуски:\n", df.isnull().sum())

features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'
df = df.dropna(subset=[target])

X = df[features]
y = df[target]

Размер датасета: (891, 15)
Пропуски:
 survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


##2. Обработка пропусков и кодирование признаков
Разделяем признаки на числовые и категориальные. Для числовых заполняем пропуски медианой и масштабируем (StandardScaler). Для категориальных заполняем пропуски самой частой категорией и применяем One-Hot Encoding. Все преобразования собираем в ColumnTransformer, а затем в общий Pipeline – это гарантирует, что обработка данных будет корректно применяться на каждом фолде кросс-валидации.

In [ ]:
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['sex', 'embarked', 'pclass']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', KNeighborsClassifier())])

##3. Разделение на обучающую и тестовую выборки
С помощью train_test_split откладываем 20% данных для финальной проверки. Параметр stratify=y сохраняет исходное соотношение выживших и погибших в обеих выборках.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

##4. Базовая модель KNN с произвольным гиперпараметром (K=3)
Создаём конвейер с K=3, обучаем на тренировочных данных и оцениваем на тестовых. Основные метрики – accuracy и F1-score, они покажут качество до подбора гиперпараметра.

In [ ]:
knn_base = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', KNeighborsClassifier(n_neighbors=3))])
knn_base.fit(X_train, y_train)
y_pred_base = knn_base.predict(X_test)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)
print("\n    Базовая модель (K=3)    ")
print(f"Accuracy: {acc_base:.4f}")
print(f"F1-score: {f1_base:.4f}")


    Базовая модель (K=3)    
Accuracy: 0.8268
F1-score: 0.7704


##5. Подбор K с помощью GridSearchCV и двух стратегий кросс-валидации
Перебираем значения K от 1 до 30. Первая стратегия – StratifiedKFold (5 фолдов с сохранением баланса классов), вторая – ShuffleSplit (5 случайных разбиений без стратификации). Для каждой выводим лучшее K и среднюю accuracy на кросс-валидации.

In [ ]:
param_grid = {'classifier__n_neighbors': np.arange(1, 31)}

# Стратегия 1: Stratified K-Fold
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_strat = GridSearchCV(pipeline, param_grid, cv=cv_stratified, scoring='accuracy', n_jobs=-1)
grid_strat.fit(X_train, y_train)

# Стратегия 2: ShuffleSplit
cv_shuffle = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
grid_shuffle = GridSearchCV(pipeline, param_grid, cv=cv_shuffle, scoring='accuracy', n_jobs=-1)
grid_shuffle.fit(X_train, y_train)

print("\n    GridSearchCV (StratifiedKFold)    ")
print("Лучший параметр K:", grid_strat.best_params_['classifier__n_neighbors'])
print("Лучшая средняя accuracy: {:.4f}".format(grid_strat.best_score_))

print("\n    GridSearchCV (ShuffleSplit)    ")
print("Лучший параметр K:", grid_shuffle.best_params_['classifier__n_neighbors'])
print("Лучшая средняя accuracy: {:.4f}".format(grid_shuffle.best_score_))


    GridSearchCV (StratifiedKFold)    
Лучший параметр K: 18
Лучшая средняя accuracy: 0.8258

    GridSearchCV (ShuffleSplit)    
Лучший параметр K: 19
Лучшая средняя accuracy: 0.8140


##6. Подбор K с помощью RandomizedSearchCV (те же стратегии)
Аналогичный поиск, но вместо полного перебора случайным образом выбираем 15 значений K. Это полезно при большом пространстве гиперпараметров; здесь просто демонстрируем работу метода.

In [ ]:
random_strat = RandomizedSearchCV(pipeline, param_grid, n_iter=15, cv=cv_stratified,
                                  scoring='accuracy', random_state=42, n_jobs=-1)
random_strat.fit(X_train, y_train)

random_shuffle = RandomizedSearchCV(pipeline, param_grid, n_iter=15, cv=cv_shuffle,
                                    scoring='accuracy', random_state=42, n_jobs=-1)
random_shuffle.fit(X_train, y_train)

print("\n    RandomizedSearchCV (StratifiedKFold)    ")
print("Лучший параметр K:", random_strat.best_params_['classifier__n_neighbors'])
print("Лучшая средняя accuracy: {:.4f}".format(random_strat.best_score_))

print("\n    RandomizedSearchCV (ShuffleSplit)    ")
print("Лучший параметр K:", random_shuffle.best_params_['classifier__n_neighbors'])
print("Лучшая средняя accuracy: {:.4f}".format(random_shuffle.best_score_))


    RandomizedSearchCV (StratifiedKFold)    
Лучший параметр K: 18
Лучшая средняя accuracy: 0.8258

    RandomizedSearchCV (ShuffleSplit)    
Лучший параметр K: 25
Лучшая средняя accuracy: 0.8112


##7. Оценка оптимальной модели на тестовых данных
Выбираем лучшую модель по результатам GridSearchCV со StratifiedKFold (обычно более стабильный вариант). Вычисляем accuracy и F1-score на тесте, выводим подробный отчёт и матрицу ошибок.

In [ ]:
best_k = grid_strat.best_params_['classifier__n_neighbors']
optimal_model = grid_strat.best_estimator_
y_pred_opt = optimal_model.predict(X_test)

acc_opt = accuracy_score(y_test, y_pred_opt)
f1_opt = f1_score(y_test, y_pred_opt)
print("\n    Оптимальная модель (K={}) на тесте    ".format(best_k))
print(f"Accuracy: {acc_opt:.4f}")
print(f"F1-score: {f1_opt:.4f}")
print("Classification report:\n", classification_report(y_test, y_pred_opt))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_opt))


    Оптимальная модель (K=18) на тесте    
Accuracy: 0.7989
F1-score: 0.7097
Classification report:
               precision    recall  f1-score   support

           0       0.80      0.90      0.85       110
           1       0.80      0.64      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179

Confusion matrix:
 [[99 11]
 [25 44]]
